In [1]:
import re
import json
import numpy as np
import pandas as pd


def analyze_results_txt(txt_path, json_path, vertical="uy"):
    """
    Analyze DSM txt output.

    Assumed DOF order:
    [ux, uy, uz, rx, ry, rz]

    vertical:
        "uy" if Y is vertical
        "uz" if Z is vertical
    """

    # ------------------------------------------------------------
    # 1) Read input JSON for section areas
    # ------------------------------------------------------------
    with open(json_path, "r", encoding="utf-8") as f:
        model = json.load(f)

    sections = model["sections"]
    elements_raw = model["elements"]

    element_section = {}
    for e_id, e_data in elements_raw.items():
        if isinstance(e_data, list):
            section = e_data[2]
        else:
            section = e_data["section"]
        element_section[int(e_id)] = section

    # ------------------------------------------------------------
    # 2) Parse txt file
    # ------------------------------------------------------------
    global_rows = []
    element_rows = []

    current_element = None

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # Global DOF row
            m = re.match(
                r"^\s*(\d+)\s+([A-Za-z_]+)\s+(\w+)\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)",
                line,
            )
            if m:
                dof = int(m.group(1))
                dof_type = m.group(2)
                status = m.group(3)
                disp = float(m.group(4))
                load = float(m.group(5))

                node_id = (dof - 1) // 6 + 1

                global_rows.append(
                    {
                        "DOF": dof,
                        "Node": node_id,
                        "Type": dof_type,
                        "Status": status,
                        "Disp": disp,
                        "Load": load,
                    }
                )

            # Element header
            m = re.match(r"^E(\d+)\s+\((.+)\)", line)
            if m:
                current_element = int(m.group(1))

            # q_local line
            if line.startswith("q_local") and current_element is not None:
                nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", line)
                q_vals = [float(x) for x in nums]

                if len(q_vals) >= 12:
                    section = element_section.get(current_element, None)
                    A = None
                    stress_MPa = None

                    if section is not None and "A" in sections[section]:
                        A = float(sections[section]["A"])

                        # axial force magnitude, kN
                        N_abs = max(abs(q_vals[0]), abs(q_vals[6]))

                        # kN / m^2 = kPa, divide by 1000 to MPa
                        stress_MPa = N_abs / A / 1000.0

                    element_rows.append(
                        {
                            "Element": current_element,
                            "Section": section,
                            "q_local": q_vals,
                            "Max_abs_element_force": max(abs(v) for v in q_vals),
                            "Axial_force_abs_kN": max(abs(q_vals[0]), abs(q_vals[6])),
                            "Area": A,
                            "Axial_stress_abs_MPa": stress_MPa,
                        }
                    )

    global_df = pd.DataFrame(global_rows)
    element_df = pd.DataFrame(element_rows)

    # ------------------------------------------------------------
    # 3) Helper functions
    # ------------------------------------------------------------
    def max_abs_row(df, col):
        idx = df[col].abs().idxmax()
        return df.loc[idx]

    # ------------------------------------------------------------
    # 4) Displacement results
    # ------------------------------------------------------------
    ux_row = max_abs_row(global_df[global_df["Type"] == "u_x"], "Disp")
    uy_row = max_abs_row(global_df[global_df["Type"] == "u_y"], "Disp")
    uz_row = max_abs_row(global_df[global_df["Type"] == "u_z"], "Disp")

    if vertical == "uy":
        vertical_row = uy_row
        transverse_row = uz_row
    elif vertical == "uz":
        vertical_row = uz_row
        transverse_row = uy_row
    else:
        raise ValueError("vertical must be 'uy' or 'uz'")

    longitudinal_row = ux_row

    # ------------------------------------------------------------
    # 5) Rotation result
    # ------------------------------------------------------------
    rot_df = global_df[global_df["Type"].isin(["theta_x", "theta_y", "theta_z"])]
    max_rotation_row = max_abs_row(rot_df, "Disp")

    # ------------------------------------------------------------
    # 6) Element force and stress
    # ------------------------------------------------------------
    max_element_force_row = max_abs_row(element_df, "Max_abs_element_force")
    max_stress_row = max_abs_row(
        element_df.dropna(subset=["Axial_stress_abs_MPa"]),
        "Axial_stress_abs_MPa",
    )

    # ------------------------------------------------------------
    # 7) Support reactions
    # ------------------------------------------------------------
    reaction_df = global_df[global_df["Status"] == "Fixed"].copy()
    max_reaction_row = max_abs_row(reaction_df, "Load")

    # ------------------------------------------------------------
    # 8) Print summary
    # ------------------------------------------------------------
    print("\n================ RESULT SUMMARY ================\n")

    print("1) Maximum longitudinal displacement")
    print(
        f"   Node {int(longitudinal_row['Node'])}, "
        f"{longitudinal_row['Type']} = {longitudinal_row['Disp']:.3f} mm"
    )

    print("\n2) Maximum vertical displacement")
    print(
        f"   Node {int(vertical_row['Node'])}, "
        f"{vertical_row['Type']} = {vertical_row['Disp']:.3f} mm"
    )

    print("\n3) Maximum transverse displacement")
    print(
        f"   Node {int(transverse_row['Node'])}, "
        f"{transverse_row['Type']} = {transverse_row['Disp']:.3f} mm"
    )

    print("\n4) Maximum rotation")
    print(
        f"   Node {int(max_rotation_row['Node'])}, "
        f"{max_rotation_row['Type']} = {max_rotation_row['Disp']:.6f} rad"
    )

    print("\n5) Maximum element force component")
    print(
        f"   Element E{int(max_element_force_row['Element'])}, "
        f"|q_local|max = {max_element_force_row['Max_abs_element_force']:.3f} kN or kN·m"
    )

    print("\n6) Maximum axial stress")
    print(
        f"   Element E{int(max_stress_row['Element'])}, "
        f"Section {max_stress_row['Section']}, "
        f"|N| = {max_stress_row['Axial_force_abs_kN']:.3f} kN, "
        f"A = {max_stress_row['Area']:.6g} m^2, "
        f"|sigma| = {max_stress_row['Axial_stress_abs_MPa']:.3f} MPa"
    )

    print("\n7) Maximum support reaction")
    print(
        f"   Node {int(max_reaction_row['Node'])}, "
        f"{max_reaction_row['Type']}, "
        f"Reaction = {max_reaction_row['Load']:.3f} kN or kN·m"
    )

    print("\n================================================\n")

    # ------------------------------------------------------------
    # 9) Return results for report use
    # ------------------------------------------------------------
    summary = {
        "max_longitudinal_displacement": longitudinal_row,
        "max_vertical_displacement": vertical_row,
        "max_transverse_displacement": transverse_row,
        "max_rotation": max_rotation_row,
        "max_element_force": max_element_force_row,
        "max_axial_stress": max_stress_row,
        "max_support_reaction": max_reaction_row,
        "global_df": global_df,
        "element_df": element_df,
    }

    return summary

In [7]:
txt_path = r"C:\Users\bchen601\Documents\GitHub\CEE6501_project_Boyang\outputs\final_structure_no_tem\final_structure_no_tem_summary.txt"
json_path = r"C:\Users\bchen601\Documents\GitHub\CEE6501_project_Boyang\inputs\final_structure.json"

summary = analyze_results_txt(
    txt_path,
    json_path,
    vertical="uy"
)


================ RESULT SUMMARY ================

1) Maximum longitudinal displacement
   Node 44, u_x = 8.250 mm

2) Maximum vertical displacement
   Node 72, u_y = -54.200 mm

3) Maximum transverse displacement
   Node 34, u_z = -294.176 mm

4) Maximum rotation
   Node 1, theta_x = 0.000455 rad

5) Maximum element force component
   Element E182, |q_local|max = 151692.404 kN or kN·m

6) Maximum axial stress
   Element E204, Section A3, |N| = 3486.523 kN, A = 30 m^2, |sigma| = 0.116 MPa

7) Maximum support reaction
   Node 76, u_y, Reaction = 17166.342 kN or kN·m




In [5]:
import re
import pandas as pd


def extract_all_support_reactions(txt_path, save_csv=True):
    rows = []

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # 匹配 GLOBAL RESULTS 里的 DOF 行
            m = re.match(
                r"^(\d+)\s+([A-Za-z_]+)\s+(\w+)\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)",
                line
            )

            if m:
                dof = int(m.group(1))
                dof_type = m.group(2)
                status = m.group(3)
                disp = float(m.group(4))
                reaction = float(m.group(5))

                node_id = (dof - 1) // 6 + 1

                if status == "Fixed":
                    rows.append({
                        "Node": node_id,
                        "DOF": dof,
                        "Type": dof_type,
                        "Prescribed displacement / rotation": disp,
                        "Reaction": reaction
                    })

    df = pd.DataFrame(rows)

    print("\n=== ALL SUPPORT REACTIONS ===")
    print(df.to_string(index=False))

    if save_csv:
        out_path = txt_path.replace("_summary.txt", "_support_reactions.csv")
        df.to_csv(out_path, index=False)
        print(f"\nSupport reactions saved to: {out_path}")

    return df

In [8]:
txt_path = r"C:\Users\bchen601\Documents\GitHub\CEE6501_project_Boyang\outputs\final_structure_no_tem\final_structure_no_tem_summary.txt"

reaction_df = extract_all_support_reactions(txt_path)


=== ALL SUPPORT REACTIONS ===
 Node  DOF Type  Prescribed displacement / rotation   Reaction
    1    1  u_x                                 0.0     -0.000
    1    2  u_y                                 0.0   2502.735
    1    3  u_z                                 0.0     -0.000
   38  224  u_y                               -50.0 -11646.340
   38  225  u_z                                 0.0      0.000
   39  230  u_y                                 0.0   2497.265
   76  452  u_y                               -50.0  17166.342

Support reactions saved to: C:\Users\bchen601\Documents\GitHub\CEE6501_project_Boyang\outputs\final_structure_no_tem\final_structure_no_tem_support_reactions.csv
